# Complaint-Type Taxonomy Classification (Negative Reviews)

Phase 1b: Semantic zero-shot classification using sentence-transformer prototypes across 10 customer complaint types.

In [1]:
import os, sys
if '..' not in sys.path:
    sys.path.insert(0, '..')
import numpy as np, pandas as pd
from src.complaints import load_negative_reviews
from src.complaint_types import (
    load_complaint_prototypes,
    build_type_vectors,
    classify_negative_reviews,
    sample_complaint_reviews,
    cross_complaint_by_product_theme,
)


In [2]:
# 1. Load data and verify embedding alignment
neg_df, neg_emb, is_aligned = load_negative_reviews(
    clean_path='../data/processed/reviews_clean.parquet',
    embeddings_path='../data/processed/embeddings.npy',
)
print('Negative reviews count:', len(neg_df))
print('Embeddings aligned:', is_aligned)


In [3]:
# 2. Load prototype sentences and embed them
prototypes = load_complaint_prototypes('../config/complaint_types.json')
type_names, type_vectors = build_type_vectors(prototypes)
print('Type vectors shape:', type_vectors.shape)


In [4]:
# 3. Classify negative reviews
assigned_types, top_sims, margins = classify_negative_reviews(
    neg_embeddings=neg_emb,
    type_names=type_names,
    type_vectors=type_vectors,
    min_similarity=0.30,
    min_margin=0.03,
)
assignments_df = pd.DataFrame({
    'review_id': neg_df['review_id'].values,
    'complaint_type': assigned_types,
    'similarity': np.round(top_sims, 4),
    'margin': np.round(margins, 4),
})
assignments_df.to_parquet('../data/processed/complaint_types.parquet', index=False)
print('Share assigned:', (assignments_df['complaint_type'] != 'Other / unclear').mean())


In [5]:
# 4. Distribution counts
counts_df = assignments_df['complaint_type'].value_counts().reset_index()
counts_df.columns = ['complaint_type', 'count']
counts_df['share'] = (counts_df['count'] / len(assignments_df) * 100).round(2).astype(str) + '%'
counts_df.to_csv('../results/complaint_type_counts.csv', index=False)
counts_df


In [6]:
# 5. Cross-tabulate with product themes
shares_table, top_pairs_df = cross_complaint_by_product_theme(
    assignments_df=assignments_df,
    topics_path='../data/processed/topics_v1_merged.parquet',
    labels_path='../data/processed/theme_labels_final.json',
    min_theme_count=100,
)
(shares_table * 100).round(2).to_csv('../results/complaint_by_theme.csv')
top_pairs_df.head(10)
